# Team 9 ML Training and Testing


*   Create a test set and a training set using the original dataset
*   Follow the steps that we use in our hands-ons to prepare the data and pipeline for training at least 3 ML models. Use any strategy that you see fit. Use N-fold cross-validation to evaluate the performance of each model.
*   Graphs for initial data exploration
*   Description of test data and test strategy you use
* Select the best performance metrics to evaluate your models for testing and explain why you choose this model
* Test your best ML model using the test set. Include a confusion matrix to show the performance of your best class ML model.





# Create a test set and a training set using the original dataset

In [65]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer


In [66]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import f1_score

In [67]:
diabetes_df = pd.read_csv('diabetes_dataset.csv')
diabetes_df.head()

,age,gender,ethnicity,education_level,income_level,employment_status,smoking_status,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,...,hdl_cholesterol,ldl_cholesterol,triglycerides,glucose_fasting,glucose_postprandial,insulin_level,hba1c,diabetes_risk_score,diabetes_stage,diagnosed_diabetes
0,58,Male,Asian,Highschool,Lower-Middle,Employed,Never,0,215,5.7,...,41,160,145,136,236,6.36,8.18,29.6,Type 2,1
1,48,Female,White,Highschool,Middle,Employed,Former,1,143,6.7,...,55,50,30,93,150,2.00,5.63,23.0,No Diabetes,0
2,60,Male,Hispanic,Highschool,Middle,Unemployed,Never,1,57,6.4,...,66,99,36,118,195,5.07,7.51,44.7,Type 2,1
3,74,Female,Black,Highschool,Low,Retired,Never,0,49,3.4,...,50,79,140,139,253,5.28,9.03,38.2,Type 2,1
4,46,Male,White,Graduate,Middle,Retired,Never,1,109,7.2,...,52,125,160,137,184,12.74,7.20,23.5,Type 2,1


In [68]:
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(diabetes_df, diabetes_df["diabetes_stage"]):
    strat_train_set = diabetes_df.loc[train_index]
    strat_test_set = diabetes_df.loc[test_index]

# Prepare the Data and Pipeline for Training at least 3 ML Models


*   Use N-fold cross-validation to evaluate the performance of each model




## Data Cleaning and Pipeline

In [69]:
diabetes_df.isnull().sum().sum()

0

In [70]:
diabetes_df.describe()

,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,family_history_diabetes,hypertension_history,cardiovascular_history,bmi,...,cholesterol_total,hdl_cholesterol,ldl_cholesterol,triglycerides,glucose_fasting,glucose_postprandial,insulin_level,hba1c,diabetes_risk_score,diagnosed_diabetes
count,100000.00000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,...,100000.000000,100000.000000,100000.000000,100000.000000,100000.00000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,50.12041,2.003670,118.911640,5.994787,6.997818,5.996468,0.219410,0.250800,0.079200,25.612653,...,185.978110,54.042790,103.000430,121.462650,111.11712,160.035050,9.061242,6.520776,30.222362,0.599980
std,15.60460,1.417779,84.409662,1.780954,1.094622,2.468406,0.413849,0.433476,0.270052,3.586705,...,32.013005,10.267374,33.390256,43.372619,13.59561,30.935472,4.954060,0.813921,9.061505,0.489904
min,18.00000,0.000000,0.000000,0.000000,3.000000,0.500000,0.000000,0.000000,0.000000,15.000000,...,100.000000,20.000000,50.000000,30.000000,60.00000,70.000000,2.000000,4.000000,2.700000,0.000000
25%,39.00000,1.000000,57.000000,4.800000,6.300000,4.300000,0.000000,0.000000,0.000000,23.200000,...,164.000000,47.000000,78.000000,91.000000,102.00000,139.000000,5.090000,5.970000,23.800000,0.000000
50%,50.00000,2.000000,100.000000,6.000000,7.000000,6.000000,0.000000,0.000000,0.000000,25.600000,...,186.000000,54.000000,102.000000,121.000000,111.00000,160.000000,8.790000,6.520000,29.000000,1.000000
75%,61.00000,3.000000,160.000000,7.200000,7.700000,7.700000,0.000000,1.000000,0.000000,28.000000,...,208.000000,61.000000,126.000000,151.000000,120.00000,181.000000,12.450000,7.070000,35.600000,1.000000
max,90.00000,10.000000,833.000000,10.000000,10.000000,16.800000,1.000000,1.000000,1.000000,39.200000,...,318.000000,98.000000,263.000000,344.000000,172.00000,287.000000,32.220000,9.800000,67.200000,1.000000


In [ ]:

#Drop the column "diabetes_stage" because we want to be able to predict this
target_col = "diabetes_stage"

X_train = strat_train_set.drop(columns=[target_col])
y_train = strat_train_set[target_col]
X_test = strat_test_set.drop(columns=[target_col])
y_test = strat_test_set[target_col]

#Split the columns into two lists, one for numericals and one for categorical
num_attribs = X_train.select_dtypes(include="number").columns.tolist()
cat_attribs = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

#pipeline for numerical columns
num_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

#pipeline for categorical columns
cat_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        ),
    ]
)

#combine the two columns again into one pipeline
full_pipeline = ColumnTransformer(
    [
        ("num", num_pipeline, num_attribs),
        ("cat", cat_pipeline, cat_attribs),
    ]
)


X_train_prepared = full_pipeline.fit_transform(X_train)
X_test_prepared = full_pipeline.transform(X_test)

print("numeric columns:", len(num_attribs))
print("categorical columns:", len(cat_attribs))


numeric columns: 24
categorical columns: 6
<class 'numpy.ndarray'>
float64


## First Model

In [72]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

log_reg_pipeline = Pipeline([
    ("preprocess", full_pipeline),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            random_state=42,
            multi_class="multinomial",
            solver="lbfgs", 
        ),
    ),
])

log_reg_pipeline.fit(X_train, y_train)

c:\Users\w\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age',
                                                   'alcohol_consumption_per_week',
                                                   'physical_activity_minutes_per_week',
                                                   'diet_score',
                                                   'sleep_hours_per_day',
                                                   'screen_time_hours_per_day',
                                                   'family_history_diabetes',
                                                   'hypertension_history',
                                                   'ca...
                                                   'diagnosed_diabetes']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['gender', 'ethnicity',
                                                   'education_level',
                                                   'income_level',
                                                   'employment_status',
                                                   'smoking_status'])])),
                ('model',
                 LogisticRegression(max_iter=1000, multi_class='multinomial',
                                    random_state=42))])

In [73]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(
    log_reg_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring="f1_macro", 
)
def display_scores(scores):
    print("Scores:", scores)
    print("Mean:", scores.mean())
    print("Standard deviation:", scores.std())

display_scores(scores)

c:\Users\w\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\w\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\w\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\w\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will b

Scores: [0.55547769 0.55708689 0.55682938 0.5557514  0.5571959 ]
Mean: 0.5564682510067973
Standard deviation: 0.0007124163718631979


## Second Model

In [90]:
from sklearn.neighbors import KNeighborsClassifier
kn = KNeighborsClassifier()
kn = kn.fit(X_train_prepared, y_train)
kn_y_pred = kn.predict(X_test_prepared)
kn_y_pred_proba = kn.predict_proba(X_test_prepared)
print("Accuracy: ", accuracy_score(y_test, kn_y_pred, normalize=True))
print("Recall: ", recall_score(y_test, kn_y_pred, average='weighted'))
print("Precision: ", precision_score(y_test, kn_y_pred, average='weighted'))
print("ROC AUC: ", roc_auc_score(y_test, kn_y_pred_proba, multi_class="ovo"))
print("F1: ", f1_score(y_test, kn_y_pred, average='weighted'))

Accuracy:  0.9443
Recall:  0.9443
Precision:  0.938101305152188
ROC AUC:  0.731298126392546
F1:  0.9389938504018349


c:\Users\w\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Third Model

# Select the best performance metrics to evaluate your models for testing and explain why you choose this model

# Test your best ML model using the test set. Include a confusion matrix to show the performance of your best class ML model.